# MS3SEG VentiMorph-RelNet V2.7 -- Ablation Study: Design, Status, Plan

**Status: designed and code-ready; NOT executed.** No ablation performance numbers exist
anywhere in this repository or the paper. Every cell below either runs real, unmodified code
from the training notebook (proving the 8 configs are correctly defined) or is explicitly
marked *pending*. Nothing here is a fabricated result.

## What the study tests

VentiMorph-RelNet V2.7 is built by adding architectural components on top of a plain
multimodal 2.5-D U-Net one at a time. The ablation isolates each addition's contribution by
re-training with only that subset of components enabled, on the same data, split, and
optimizer settings.

| Config | Multimodal (T1+T2) | Context block | Ventricle-relation guidance | Prototype bank | Uncertainty head | What it isolates |
|---|:---:|:---:|:---:|:---:|:---:|---|
| `A0_flair_2p5d` | - | - | none | - | - | Baseline: FLAIR-only 2.5-D U-Net |
| `A1_multimodal` | Y | - | none | - | - | + registered T1/T2 via modality-gated fusion |
| `A2_context` | Y | Y | none | - | - | + Adaptive Anisotropic Context (dilated bottleneck) |
| `A3_vent_aux` | Y | Y | vent | - | - | + a plain ventricle auxiliary head (no distance/direction) |
| `A4_distance` | Y | Y | distance | - | - | + near/mid/far ventricle-distance bands |
| `A5_directional` | Y | Y | directional | - | - | + radial/tangential orientation relative to the ventricle |
| `A6_prototypes` | Y | Y | directional | Y | - | + relation-conditioned nWMH/abWMH prototype bank |
| `A7_full` | Y | Y | directional | Y | Y | Full model (as reported everywhere else in this repo) |


## The real ablation configuration code

`ExperimentConfig`, unmodified, copied from `notebook381a28a764(2).ipynb` (needed to construct
the 8 configs below).


In [1]:
from dataclasses import dataclass, asdict
import pandas as pd


@dataclass
class ExperimentConfig:
    epochs: int = 60
    lr: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 12
    min_improvement: float = 1e-4
    grad_clip: float = 2.0

    plateau_factor: float = 0.5
    plateau_patience: int = 3
    minimum_lr: float = 1e-6

    collapse_margin: float = 0.05
    collapse_patience: int = 2

    gradient_alarm_threshold: float = 300.0
    gradient_alarm_patience: int = 2

    rollback_lr_factor: float = 0.5
    max_rollbacks: int = 2

    transfer_every_epochs: int = 5

    base: int = 24
    use_multimodal: bool = True
    use_context: bool = True
    guidance_mode: str = "directional"
    use_prototypes: bool = True
    use_uncertainty: bool = True

    fold: int = 0
    seed: int = 42
    auto_resume: bool = True


print("ExperimentConfig defined.")


ExperimentConfig defined.


This is the **exact, unmodified** ablation cell from `notebook381a28a764(2).ipynb` -- same
variable names, same logic, same `RUN_ABLATIONS = False`. It is included verbatim (not
re-typed or summarized) so there is no ambiguity about what the actual code does.


In [2]:
ABLATIONS = {
    "A0_flair_2p5d": ExperimentConfig(use_multimodal=False, use_context=False, guidance_mode="none", use_prototypes=False, use_uncertainty=False),
    "A1_multimodal": ExperimentConfig(use_context=False, guidance_mode="none", use_prototypes=False, use_uncertainty=False),
    "A2_context": ExperimentConfig(guidance_mode="none", use_prototypes=False, use_uncertainty=False),
    "A3_vent_aux": ExperimentConfig(guidance_mode="vent", use_prototypes=False, use_uncertainty=False),
    "A4_distance": ExperimentConfig(guidance_mode="distance", use_prototypes=False, use_uncertainty=False),
    "A5_directional": ExperimentConfig(guidance_mode="directional", use_prototypes=False, use_uncertainty=False),
    "A6_prototypes": ExperimentConfig(guidance_mode="directional", use_prototypes=True, use_uncertainty=False),
    "A7_full": ExperimentConfig(),
}

# Each config otherwise defaults to epochs=60 / patience=12 (the full-training budget), which
# would make this 8-way ablation cost roughly 8x a full fold's training time. ABLATION_MAX_EPOCHS
# caps every ablation run to a smaller, still-decisive budget -- the stability pilot earlier in
# the training notebook already shows all three foreground classes separating within ~10-20
# epochs. Raise it (and your Kaggle GPU-hour budget) for a stricter, slower study.
ABLATION_MAX_EPOCHS = 12
ABLATION_PATIENCE = 6

for cfg in ABLATIONS.values():
    cfg.epochs = ABLATION_MAX_EPOCHS
    cfg.patience = ABLATION_PATIENCE

RUN_ABLATIONS = False  # deferred: an 8-config study is ~14-15 GPU-hours; not run for this paper due to time constraints
if RUN_ABLATIONS and not GPU_COMPATIBLE:
    raise RuntimeError(
        "RUN_ABLATIONS requires a compatible CUDA GPU. Select T4 or another "
        "supported Kaggle GPU, restart the session, and Run All."
    )

if RUN_ABLATIONS:
    estimated_hours = len(ABLATIONS) * ABLATION_MAX_EPOCHS * 9 / 60
    print(
        f"Running {len(ABLATIONS)} ablation configs, {ABLATION_MAX_EPOCHS} epochs each "
        f"(roughly {estimated_hours:.1f} GPU-hours total on a T4 at ~9 min/epoch). "
        "Uses Fold-0's train/val split only; the locked test set is never touched."
    )
    summary = []
    for name, cfg in ABLATIONS.items():
        print("="*80, "\n", name)
        m, h, d = train_experiment(cfg, name)
        vr = evaluate_patients(m, fold_val_ids, f"{name}_validation")
        summary.append({
            "experiment": name,
            "dice_vent": vr.dice_vent.mean(),
            "dice_nwmh": vr.dice_nwmh.mean(),
            "dice_abwmh": vr.dice_abwmh.mean(),
            "hd95_abwmh_mm": vr.hd95_abwmh_mm.mean(),
            "lesion_f1": vr.abwmh_lesion_f1.mean(),
            "periventricular_fp_rate": vr.periventricular_fp_rate.mean(),
        })
        del m
        gc.collect()
        torch.cuda.empty_cache()
    ablation_summary_df = pd.DataFrame(summary)
    ablation_summary_df.to_csv(OUTPUT_DIR / "ablation_summary.csv", index=False)
    display(ablation_summary_df)

print("RUN_ABLATIONS =", RUN_ABLATIONS, "-> the training/evaluation block above did not execute.")


RUN_ABLATIONS = False -> the training/evaluation block above did not execute.


## Confirm the 8 configs are correctly defined

This inspects the real `ABLATIONS` dict built above -- the actual architectural switches each
config will train with, once run. No performance numbers; this only proves the configuration
logic itself is correct.


In [3]:
config_table = pd.DataFrame({
    name: {
        "use_multimodal": cfg.use_multimodal,
        "use_context": cfg.use_context,
        "guidance_mode": cfg.guidance_mode,
        "use_prototypes": cfg.use_prototypes,
        "use_uncertainty": cfg.use_uncertainty,
        "epochs (capped)": cfg.epochs,
        "patience (capped)": cfg.patience,
    }
    for name, cfg in ABLATIONS.items()
}).T
config_table


,use_multimodal,use_context,guidance_mode,use_prototypes,use_uncertainty,epochs (capped),patience (capped)
A0_flair_2p5d,False,False,none,False,False,12,6
A1_multimodal,True,False,none,False,False,12,6
A2_context,True,True,none,False,False,12,6
A3_vent_aux,True,True,vent,False,False,12,6
A4_distance,True,True,distance,False,False,12,6
A5_directional,True,True,directional,False,False,12,6
A6_prototypes,True,True,directional,True,False,12,6
A7_full,True,True,directional,True,True,12,6


## Why it has not been run

`RUN_ABLATIONS = False` (confirmed above). Even at the reduced 12-epoch budget, training all 8
configs costs an estimated **~14-15 GPU-hours** on a Kaggle T4 (8 configs x 12 epochs x ~9
min/epoch) -- more time than was available before submission.


In [4]:
estimated_hours = len(ABLATIONS) * ABLATION_MAX_EPOCHS * 9 / 60
print(f"{len(ABLATIONS)} configs x {ABLATION_MAX_EPOCHS} epochs x ~9 min/epoch "
      f"= approximately {estimated_hours:.1f} GPU-hours on a Kaggle T4.")


8 configs x 12 epochs x ~9 min/epoch = approximately 14.4 GPU-hours on a Kaggle T4.


## Result table (template -- to be filled in once run, not before)

Columns match exactly what `evaluate_patients()` already computes and what
`ablation_summary.csv` will contain once the cell above is set to `RUN_ABLATIONS = True` and
actually executed on Kaggle. Every cell below is the literal string `"pending"` -- not a
placeholder number, not an estimate.


In [5]:
result_columns = [
    "ventricle_dice", "nwmh_dice", "abwmh_dice",
    "hd95_abwmh_mm", "lesion_f1_abwmh", "periventricular_fp_rate",
]

ablation_result_template = pd.DataFrame(
    "pending", index=list(ABLATIONS.keys()), columns=result_columns
)
ablation_result_template


,ventricle_dice,nwmh_dice,abwmh_dice,hd95_abwmh_mm,lesion_f1_abwmh,periventricular_fp_rate
A0_flair_2p5d,pending,pending,pending,pending,pending,pending
A1_multimodal,pending,pending,pending,pending,pending,pending
A2_context,pending,pending,pending,pending,pending,pending
A3_vent_aux,pending,pending,pending,pending,pending,pending
A4_distance,pending,pending,pending,pending,pending,pending
A5_directional,pending,pending,pending,pending,pending,pending
A6_prototypes,pending,pending,pending,pending,pending,pending
A7_full,pending,pending,pending,pending,pending,pending


## Plan to complete it

1. Open `notebook381a28a764(2).ipynb` on Kaggle (same `MS3SEG` + `ms3seg-v27-best-checkpoint`
   inputs already used for everything else).
2. Set `RUN_ABLATIONS = True`.
3. Run All. It iterates configs A0->A7 sequentially and is **not resumable mid-ablation** --
   budget an uninterrupted block of time, or expect to restart from A0 if the session drops.
   Consider lowering `ABLATION_MAX_EPOCHS` further (e.g. to 8) for a cheaper first pass if
   GPU-hours are tight; the stability pilot elsewhere in that notebook shows foreground classes
   separating within ~10 epochs, so even a short budget should be directionally informative.
4. It writes `ablation_summary.csv` to the Kaggle working directory -- download it, replace the
   `"pending"` cells in the template above with the real numbers, and update
   `docs/RESULTS_MAP.md` section 5 to move this item from "not run" to "executed."
